# Phase 3 — Synonyms, on `benzon:synonyms`

`plan_benzon.md` Part 3 / `plan_benzon_implementation.md` Phase 3, extended past its own
minimal spec. The plan's own step 4 already calls for one shared `run_multi_sentiment` pass
over three sentiments (confidence, commitment, nuance) on `benzon:synonyms` — this notebook
keeps that shared-pass structure but, rather than picking just one winning wording each for
commitment and nuance, tests **every OM** (Observation Method — renamed from "follow-up
variant" in Phases 1/2; the specific self-report wording tested) each construct has, crossed
against every applicable ground truth, on this third dataset — the same full-matrix
philosophy `1_commitment.ipynb`/`2_nuance.ipynb` used internally, now spanning all
three constructs studied so far in one combined table:
**(confidence, commitment, nuance) × GT × OM**.

**Dataset** (`vconf/benzon_data.py`'s `load_synonym_pairs`): 30 (name1, name2) pairs, each
asked via all 4 `SYNONYM_TEMPLATES` ("Is X Y?" / "Are X and Y the same thing?" / "Is X exactly
Y?" / the negation control "Is X not Y?") — 120 items total, in three relation tiers: 13
WordNet-verified exact-synonym pairs, 9 common-name/scientific-name pairs, and 8 hand-curated
*caveated* pairs (Benzon's own salt/NaCl-style near-synonyms, each with a real "well,
technically..." asterisk). Every pair names the same referent by construction, so the first
three templates' expected answer is always "Yes" and the negation template's is always "No".

**Nine OMs across three constructs**, in the order the tables below use, one shared Phase-0
pass:

1. `confidence` (`sentiment.CONFIDENCE`) — the paper's own default; no wording variants were
   ever tested for this construct, so it's the sole OM for the `confidence` row.
2. `commitment` (`sentiment.MACHINE_COMMITMENT`)
3. `commitment_defined` (`sentiment.MACHINE_COMMITMENT_DEFINED`)
4. `commitment_parallel` (`sentiment.MACHINE_COMMITMENT_PARALLEL`)
5. `commitment_challenge` (`sentiment.NATURAL_COMMITMENT`)
6. `nuance` (`sentiment.NUANCE`)
7. `nuance_defined` (`sentiment.NUANCE_DEFINED`)
8. `nuance_ambiguity` (`sentiment.NUANCE_AMBIGUITY`)
9. `nuance_certainty` (`sentiment.NUANCE_CERTAINTY`)

**Six ground truths**, every one computed once and reused across all nine OMs (none of them
depend on which follow-up phrasing is being tested), in the order the tables below use:
labeled correctness, answer logit, self challenge resistance, labeled challenge resistance, binary entropy, nonbinary logit.

- **labeled correctness** — `ground_truth.SynonymAnswerKey`, the plan's own boolean polarity-match
  grader (`detect_yes_no_polarity` against `meta["expected_answer"]`), called directly against
  the real `QuestionItem`s rather than through `notebook.graded` — `graded`'s generic bridge
  reconstructs a bare `QuestionItem` with no `meta` at all, which `SynonymAnswerKey` needs.
- **answer logit** — `metrics.mean_answer_logprob`, free, always available.
- **self challenge resistance** and **labeled challenge resistance** — `commitment_challenge.natural_commitment_challenge_metric`,
  extended here to a *third* dataset. Correctness detection already generalizes for free
  (`_model_answered_correctly`'s Yes/No-polarity branch, built for ontology-trivials in Phase
  1, applies unchanged now that `load_synonym_pairs` also mirrors its boolean answer onto
  literal `("Yes",)`/`("No",)` gold-answer text). labeled challenge resistance needed one new hand-authored table,
  `commitment_challenge.HANDCRAFTED_COUNTERFEITS_SYNONYMS` — but since every pair in this
  dataset names the same referent by construction, the counterfeit claim is always the same
  underlying assertion ("X and Y are actually different things") regardless of which of the 4
  templates asked about it, so only 30 claims needed hand-authoring (one per pair), not 120.
- **binary entropy** — `metrics.answer_set_entropy`, `nuance`'s ground truth from Phase 2,
  dataset-agnostic by construction.
- **nonbinary logit** (Non-Binary Ground Truth) — a `nuance`-flavored ground truth distinct from entropy
  GT: rather than uncertainty *between* Yes and No, it reads how much probability mass the
  model's own next-token distribution puts *outside* the binary {Yes, No} frame entirely, at
  the exact Phase-0 answer position — `1 - P(Yes) - P(No)`.

**Phase 3's own gate** (`plan_benzon_implementation.md`): mean `nuance`-entropy is measurably
higher on the hand-curated *caveated* pairs than on the *clean* (exact-synonym) pairs — a
concrete, falsifiable comparison, checked near the end of this notebook.

In [1]:
import json
import os
import pathlib
import sys

import matplotlib
import numpy as np
import pandas as pd

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "vconf").is_dir())
sys.path.insert(0, str(ROOT))

HERE = pathlib.Path.cwd()
with open(HERE / "config.json") as f:
    MODEL_CONFIG = json.load(f)
MODEL_NAME = MODEL_CONFIG["model"]
os.environ["VCONF_MODEL"] = MODEL_NAME
CACHE_DIR = HERE / "cache" / MODEL_NAME
OUT_DIR = HERE / "out" / MODEL_NAME
FIGS_DIR = OUT_DIR / "figs"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
FIGS_DIR.mkdir(parents=True, exist_ok=True)
TRIAL_LOG_PATH = CACHE_DIR / "trial.json"
CONSTRUCT_PATH = CACHE_DIR / "construct.json"
SCALE = MODEL_CONFIG.get("scale", {})

from vconf import notebook as nb

_run_config = nb.run_config


def _run_config_with_overrides(base="gemma-categorical", **overrides):
    cfg = _run_config(base, **overrides)
    if MODEL_CONFIG.get("attn_implementation"):
        cfg = cfg.scaled(attn_implementation=MODEL_CONFIG["attn_implementation"])
    return cfg


nb.run_config = _run_config_with_overrides
from vconf import commitment_challenge as CC
from vconf import data as datamod
from vconf import ground_truth as GT
from vconf import metrics as M
from vconf import pipeline
from vconf.sentiment import (
    CONFIDENCE,
    MACHINE_COMMITMENT, MACHINE_COMMITMENT_PARALLEL, MACHINE_COMMITMENT_DEFINED, NATURAL_COMMITMENT,
    NUANCE, NUANCE_AMBIGUITY, NUANCE_CERTAINTY, NUANCE_DEFINED,
)

cfg = nb.run_config("gemma-categorical", dataset="benzon:synonyms", name="synonyms")
print(nb.describe(cfg))

profile          : reduced
model            : Qwen/Qwen2.5-7B-Instruct (28 layers)
sentiment        : confidence  (ground truth: correctness)
prompt / dataset : categorical / benzon:synonyms
layer sweep      : (0, 5, 11, 16, 22, 27)
trial counts     : {'steering': 24, 'patching': 24, 'noising': 32, 'swap': 24, 'attention': 24}
activation set   : 300   calibration set: 40
chat template    : True   attention impl: None
NOTE             : reduced profile — procedures, prompts, positions and
                   metrics follow the manual exactly, but the model and the
                   sample sizes are smaller than the paper's, so the numbers
                   here are not expected to match its reported values.


## Dataset

30 (name1, name2) pairs × 4 `SYNONYM_TEMPLATES` = 120 items, in three relation tiers.

In [2]:
items = datamod.load_dataset_items("benzon:synonyms", limit=SCALE.get("synonyms"))
print(f"{len(items)} synonym-pair questions")
display(pd.Series([item.meta["relation"] for item in items]).value_counts().to_frame("count"))
pd.DataFrame(
    [{"qid": i.qid, "question": i.question, "relation": i.meta["relation"]} for i in items]
).head(8)

120 synonym-pair questions


,count
exact_synonym,52
common_scientific,36
caveated,32


,qid,question,relation
0,synonym_0_0,Is a car an automobile?,exact_synonym
1,synonym_0_1,Are a car and an automobile the same thing?,exact_synonym
2,synonym_0_2,Is a car exactly an automobile?,exact_synonym
3,synonym_0_3,Is a car not an automobile?,exact_synonym
4,synonym_1_0,Is a couch a sofa?,exact_synonym
5,synonym_1_1,Are a couch and a sofa the same thing?,exact_synonym
6,synonym_1_2,Is a couch exactly a sofa?,exact_synonym
7,synonym_1_3,Is a couch not a sofa?,exact_synonym


In [3]:
loaded = nb.open_model(cfg, device_map=MODEL_CONFIG.get("device_map"))

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

## Nine OMs across three constructs, one shared Phase-0 pass

`CONSTRUCT_OF` tags each OM with which of the three top-level constructs it belongs to, used
below to group the full matrix's rows.

In [4]:
OMS = {
    "confidence": CONFIDENCE,
    "commitment": MACHINE_COMMITMENT,
    "commitment_defined": MACHINE_COMMITMENT_DEFINED,
    "commitment_parallel": MACHINE_COMMITMENT_PARALLEL,
    "commitment_challenge": NATURAL_COMMITMENT,
    "nuance": NUANCE,
    "nuance_defined": NUANCE_DEFINED,
    "nuance_ambiguity": NUANCE_AMBIGUITY,
    "nuance_certainty": NUANCE_CERTAINTY,
}
CONSTRUCT_OF = {
    "confidence": "confidence",
    "commitment": "commitment",
    "commitment_defined": "commitment",
    "commitment_parallel": "commitment",
    "commitment_challenge": "commitment",
    "nuance": "nuance",
    "nuance_defined": "nuance",
    "nuance_ambiguity": "nuance",
    "nuance_certainty": "nuance",
}

out = pipeline.run_multi_sentiment(loaded, items, cfg, OMS)
by_qid_variant = {name: {t.qid: t for t in pipeline.filter_valid(trials)} for name, trials in out.items()}
for name, d in by_qid_variant.items():
    print(f"{len(d)}/{len(out[name])} {name} trials valid")

common_qids = sorted(set.intersection(*(set(d) for d in by_qid_variant.values())))
print(f"\n{len(common_qids)} qids valid under all nine OMs")

120/120 confidence trials valid
120/120 commitment trials valid
120/120 commitment_defined trials valid
120/120 commitment_parallel trials valid
120/120 commitment_challenge trials valid
120/120 nuance trials valid
120/120 nuance_defined trials valid
120/120 nuance_ambiguity trials valid
120/120 nuance_certainty trials valid

120 qids valid under all nine OMs


## Six ground truths

answer logit and binary entropy are free/cheap (a forward pass each); self challenge resistance/labeled challenge resistance need the
evidence/counterfeit challenge run per trial — the expensive step. All six depend only on the
question/answer, not on which OM is being tested, so each is computed once and reused across
all nine OMs below.

**nonbinary logit** (Non-Binary Ground Truth) — a `nuance`-flavored ground truth distinct from
`entropy_gt`: rather than uncertainty *between* Yes and No, it reads how much probability mass
the model's own next-token distribution puts *outside* the binary {Yes, No} frame entirely, at
the exact Phase-0 answer position (no forced-word suffix, no generation) — `1 - P(Yes) -
P(No)`. A model that wants to hedge, qualify, or give a third kind of answer ("well,
technically...", "it depends", "sort of") rather than commit to a clean binary reads high
here; a model confidently picking Yes-or-No reads near zero. Every question in this dataset
genuinely is a yes/no question ("Is X Y?"), so the {Yes, No} frame is always the right frame to
measure leakage from.

In [5]:
from vconf.metrics import nonbinary_mass  # promoted from this notebook into vconf/metrics.py — reusable nonbinary logit

items_by_qid = {it.qid: it for it in items}
any_trial_by_qid = by_qid_variant["confidence"]  # same question/answer across all nine OMs

logit_gt_by_qid = {q: M.mean_answer_logprob(any_trial_by_qid[q]) for q in common_qids}
logit_gt = np.array([logit_gt_by_qid[q] for q in common_qids])

synonym_key = GT.SynonymAnswerKey()
correctness_by_qid = {}
unparseable = 0
for q in common_qids:
    trial = any_trial_by_qid[q]
    label = synonym_key.label(items_by_qid[q], trial)
    if trial.note == "unparseable polarity":
        unparseable += 1
        correctness_by_qid[q] = float("nan")
    else:
        correctness_by_qid[q] = float(label)
correctness_gt = np.array([correctness_by_qid[q] for q in common_qids])

entropy_by_qid = {q: M.answer_set_entropy(any_trial_by_qid[q], loaded, cfg) for q in common_qids}
entropy_gt = np.array([entropy_by_qid[q] for q in common_qids])

nbgt_by_qid = {q: nonbinary_mass(any_trial_by_qid[q], loaded, cfg) for q in common_qids}
nbgt = np.array([nbgt_by_qid[q] for q in common_qids])

challenge_kinds = {}
gegt_by_qid = {}
hegt_by_qid = {}
for q in common_qids:
    trial = any_trial_by_qid[q]
    challenge = CC.build_natural_commitment_challenge(trial, loaded, cfg)
    challenge_kinds[q] = challenge["kind"]
    gegt_by_qid[q] = CC.natural_commitment_challenge_metric(trial, loaded, cfg, source="generated")
    # genuine-branch trials don't touch the counterfeit claim at all, so self challenge resistance and labeled challenge resistance
    # are identical there — only counterfeit-branch trials need a second, separate pass.
    hegt_by_qid[q] = (
        gegt_by_qid[q] if challenge["kind"] == "genuine"
        else CC.natural_commitment_challenge_metric(trial, loaded, cfg, source="handcrafted")
    )
gegt = np.array([gegt_by_qid[q] for q in common_qids])
hegt = np.array([hegt_by_qid[q] for q in common_qids])

display(pd.Series(challenge_kinds.values()).value_counts().to_frame("count"))
print(f"\n{unparseable} unparseable answers excluded from the labeled correctness GT")
print(f"labeled correctness  mean={np.nanmean(correctness_gt):.3f}")
print(f"answer logit     mean={logit_gt.mean():.3f} std={logit_gt.std():.3f}")
print(f"self challenge resistance         mean={gegt.mean():.3f} std={gegt.std():.3f}")
print(f"labeled challenge resistance         mean={hegt.mean():.3f} std={hegt.std():.3f}")
print(f"binary entropy   mean={entropy_gt.mean():.3f} std={entropy_gt.std():.3f}")
print(f"nonbinary logit         mean={nbgt.mean():.3f} std={nbgt.std():.3f}")

,count
counterfeit,77
genuine,43



2 unparseable answers excluded from the labeled correctness GT
labeled correctness  mean=0.653
answer logit     mean=-0.108 std=0.074
self challenge resistance         mean=2.476 std=2.179
labeled challenge resistance         mean=1.706 std=2.807
binary entropy   mean=0.453 std=0.294
nonbinary logit         mean=0.086 std=0.165


## Full matrix: (confidence, commitment, nuance) × GT × OM

Every OM's self-report correlated against all six ground truths — rows grouped by which
construct the OM belongs to (`CONSTRUCT_OF`), so the table reads as three stacked blocks
rather than nine unordered rows.

In [6]:
selfreport_by_om = {
    name: np.array([spec.class_midpoint[spec.classes[by_qid_variant[name][q].class_index]] for q in common_qids])
    for name, spec in OMS.items()
}

gt_columns = {
    "labeled correctness": correctness_gt,
    "answer logit": logit_gt,
    "self challenge resistance": gegt,
    "labeled challenge resistance": hegt,
    "binary entropy": entropy_gt,
    "nonbinary logit": nbgt,
}

rho_matrix = pd.DataFrame({
    gt_name: {name: M.intrinsic_correlation(sr, gt_values) for name, sr in selfreport_by_om.items()}
    for gt_name, gt_values in gt_columns.items()
})
rho_matrix = rho_matrix.reindex(OMS.keys())
rho_matrix.index = pd.MultiIndex.from_tuples(
    [(CONSTRUCT_OF[name], name) for name in rho_matrix.index], names=["construct", "OM"]
)

_heatmap_cmap = matplotlib.colormaps["coolwarm"].copy()
_heatmap_cmap.set_bad(color="#f0f0f0")  # NaN (collapsed self-reports) as light gray, not black
display(
    rho_matrix.style
        .format("{:.3f}", na_rep="—")
        .background_gradient(cmap=_heatmap_cmap, vmin=-1, vmax=1, axis=None)
)

/home/stud_homes/s7846062/fatass/home/thesis/experiment/code_morph/vconf/metrics.py:202: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return float(stats.spearmanr(x[mask], y[mask]).correlation)


In [7]:
for construct in ("confidence", "commitment", "nuance"):
    sub = rho_matrix.loc[construct]
    print(f"{construct}:")
    for gt_name in gt_columns:
        winner = sub[gt_name].idxmax()
        print(f"  winner vs. {gt_name:12s}: {winner} (rho={sub.loc[winner, gt_name]:.3f})")
    print()

confidence:
  winner vs. labeled correctness: confidence (rho=0.528)
  winner vs. answer logit: confidence (rho=0.544)
  winner vs. self challenge resistance: confidence (rho=-0.132)
  winner vs. labeled challenge resistance: confidence (rho=-0.228)
  winner vs. binary entropy: confidence (rho=-0.291)
  winner vs. nonbinary logit: confidence (rho=-0.526)

commitment:
  winner vs. labeled correctness: commitment_challenge (rho=0.149)
  winner vs. answer logit: commitment_challenge (rho=0.347)
  winner vs. self challenge resistance: commitment_challenge (rho=-0.051)
  winner vs. labeled challenge resistance: commitment_challenge (rho=-0.016)
  winner vs. binary entropy: commitment_challenge (rho=-0.180)
  winner vs. nonbinary logit: commitment_challenge (rho=-0.404)

nuance:
  winner vs. labeled correctness: nuance_ambiguity (rho=-0.482)
  winner vs. answer logit: nuance_ambiguity (rho=-0.435)
  winner vs. self challenge resistance: nuance_defined (rho=0.137)
  winner vs. labeled challen

## Class distributions

In [8]:
distribution_by_om = {}
top_share_by_om = {}
for name, spec in OMS.items():
    dist = M.class_histogram(
        np.array([by_qid_variant[name][q].class_index for q in common_qids]), classes=spec.classes
    )
    distribution_by_om[name] = dist
    top_share_by_om[name] = max(dist.values()) / len(common_qids)

display(pd.DataFrame(distribution_by_om).T)
print("top class share:", {name: f"{s:.1%}" for name, s in top_share_by_om.items()})

,No chance,Really unlikely,Chances are slight,Unlikely,Less than even,Better than even,Likely,Very good chance,Highly likely,Almost certain,...,Slightly committed,Somewhat committed,Moderately committed,Mostly committed,Highly committed,Fully committed,Flat,Somewhat nuanced,Nuanced,Highly nuanced
confidence,0.0,0.0,0.0,1.0,0.0,1.0,68.0,0.0,20.0,30.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
commitment,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,120.0,NaN,NaN,NaN,NaN
commitment_defined,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,120.0,NaN,NaN,NaN,NaN
commitment_parallel,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,120.0,NaN,NaN,NaN,NaN
commitment_challenge,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,103.0,NaN,NaN,NaN,NaN
nuance,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,36.0,12.0,72.0,0.0
nuance_defined,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,50.0,9.0,61.0,0.0
nuance_ambiguity,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,29.0,12.0,79.0,0.0
nuance_certainty,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,69.0,2.0,49.0,0.0


top class share: {'confidence': '56.7%', 'commitment': '100.0%', 'commitment_defined': '100.0%', 'commitment_parallel': '100.0%', 'commitment_challenge': '85.8%', 'nuance': '60.0%', 'nuance_defined': '50.8%', 'nuance_ambiguity': '65.8%', 'nuance_certainty': '57.5%'}


## Phase 3 gate (`plan_benzon_implementation.md`)

> mean `nuance`-ground-truth entropy is measurably higher on hand-curated caveated pairs than
> on clean exact-synonym pairs.

A stop-and-fix gate, not a checkpoint to note and continue past — the caveated-vs-clean
comparison is Phase 3's whole reason for keeping the hand-curated tier at all.

In [9]:
relation_by_qid = {it.qid: it.meta["relation"] for it in items}
entropy_df = pd.DataFrame({
    "qid": common_qids,
    "relation": [relation_by_qid[q] for q in common_qids],
    "entropy": [entropy_by_qid[q] for q in common_qids],
})
display(entropy_df.groupby("relation")["entropy"].agg(["mean", "count"]))

caveated_mean = entropy_df.loc[entropy_df.relation == "caveated", "entropy"].mean()
clean_mean = entropy_df.loc[entropy_df.relation == "exact_synonym", "entropy"].mean()
print(f"\ncaveated mean entropy: {caveated_mean:.3f}   exact_synonym (clean) mean entropy: {clean_mean:.3f}")

checks = {
    "caveated pairs show higher nuance-entropy than clean exact-synonym pairs": caveated_mean > clean_mean,
}
for name, ok in checks.items():
    print(f"{'PASS' if ok else 'FAIL'}  {name}")

gate_passed = all(checks.values())
print(f"\n{'GATE PASSED' if gate_passed else 'GATE FAILED'}"
      f" — {'caveated pairs read as measurably less certain, as expected' if gate_passed else 'debug before treating this dataset as validated'}")

,mean,count
relation,,
caveated,0.407476,32
common_scientific,0.475038,36
exact_synonym,0.465271,52



caveated mean entropy: 0.407   exact_synonym (clean) mean entropy: 0.465
FAIL  caveated pairs show higher nuance-entropy than clean exact-synonym pairs

GATE FAILED — debug before treating this dataset as validated


**Interpretation.** The gate above is Phase 3's own falsifiable check, independent of which OM
wins the full matrix above — it's a property of the dataset/model (via the binary entropy), not of
any one self-report wording. The full matrix is the richer result: which OM tracks which
ground truth best, per construct, on a dataset genuinely different in kind from TriviaQA and
ontology-trivials (caveated near-synonyms rather than trivia facts or uncontested is-a
statements) — a third, independent check on whether Phase 1/2's winning wordings generalize,
alongside `2_nuance.ipynb`'s own ontology-trivials secondary check.

## Log every cell to `trial.json`

Same shared calibration log as `1_commitment.ipynb`/`2_nuance.ipynb` — this is the one
notebook with the *complete* 9×6 matrix, so it fills in the most cells at once.

In [10]:
from vconf import trial_log as TL

records = []
for om, row in rho_matrix.droplevel("construct").iterrows():
    for gt_display, gt_key in TL.GT_KEY.items():
        records.append({"dataset": "synonyms", "om": om, "gt": gt_key, "rho": row[gt_display], "n": len(common_qids)})

TL.upsert(records, path=TRIAL_LOG_PATH)
print(f"logged {len(records)} cells to {TRIAL_LOG_PATH}")

logged 54 cells to /home/stud_homes/s7846062/fatass/home/thesis/experiment/code_morph/notebooks_benzon/phase_0_calibration/cache/qwen/trial.json
